# ETL Pipeline Exploration

This notebook documents the exploratory development of our ETL (Extract, Transform, Load) pipeline, prototyped on a single ticker (AAPL) before being generalized into `src/etl.py` for multi-ticker processing.

**Pipeline steps:**
1. Load raw data (prices + quarterly fundamentals)
2. Point-in-Time join (align fundamentals to daily prices by publication date)
3. Feature engineering (trend, momentum, volatility, fundamental indicators)
4. Target engineering (next-day log-return + binary direction)
5. Cleaning and validation

In [1]:
import polars as pl

# Load raw datasets
df_prices = pl.read_parquet("../data/raw/prices.parquet")
df_income = pl.read_parquet("../data/raw/income.parquet")
df_balance = pl.read_parquet("../data/raw/balance.parquet")

# Filter to AAPL for prototyping
ticker = "AAPL"
prices = df_prices.filter(pl.col("Ticker") == ticker).sort("Date")
print(f"Prices: {prices.shape}")
print(f"Income statements: {df_income.filter(pl.col('Ticker') == ticker).shape}")
print(f"Balance sheets: {df_balance.filter(pl.col('Ticker') == ticker).shape}")
prices.head()

Prices: (1238, 11)
Income statements: (19, 28)
Balance sheets: (19, 30)


Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding
str,i64,date,f64,f64,f64,f64,f64,i64,f64,i64
"""AAPL""",111052,2020-04-13,67.08,68.42,66.46,68.31,66.03,131022924,null,17295948000
"""AAPL""",111052,2020-04-14,70.0,72.06,69.51,71.76,69.37,194994688,null,17295948000
"""AAPL""",111052,2020-04-15,70.6,71.58,70.16,71.11,68.73,131154564,null,17295948000
"""AAPL""",111052,2020-04-16,71.84,72.05,70.59,71.67,69.28,157125160,null,17295948000
"""AAPL""",111052,2020-04-17,71.17,71.74,69.22,70.7,68.34,215249912,null,17337340000


## Point-in-Time Join

Quarterly fundamentals must be aligned to daily prices using the **Publish Date** (not Report Date). This ensures the model only sees information that was publicly available on each trading day, preventing look-ahead bias.

In [2]:
# Merge income + balance sheets on Ticker and Report Date
df_fund = df_income.join(
    df_balance,
    on=["Ticker", "Report Date"],
    suffix="_bal"
).sort(["Ticker", "Publish Date"])

# Filter fundamentals for our ticker
fund = df_fund.filter(pl.col("Ticker") == ticker).sort("Publish Date")

# Point-in-Time join: for each trading day, attach the most recent public fundamental data
df = prices.join_asof(
    fund,
    left_on="Date",
    right_on="Publish Date",
    by="Ticker"
)
print(f"Joined dataset: {df.shape}")
df.tail()

Joined dataset: (1238, 66)


/var/folders/dn/gclqbh253nl2n8zr7mlqnqs40000gn/T/ipykernel_24477/3349581532.py:12: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df = prices.join_asof(


Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding,SimFinId_right,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Revenue,Cost of Revenue,Gross Profit,Operating Expenses,"Selling, General & Administrative",Research & Development,Depreciation & Amortization,Operating Income (Loss),Non-Operating Income (Loss),"Interest Expense, Net","Pretax Income (Loss), Adj.",Abnormal Gains (Losses),Pretax Income (Loss),"Income Tax (Expense) Benefit, Net",Income (Loss) from Continuing Operations,Net Extraordinary Gains (Losses),Net Income,Net Income (Common),SimFinId_bal,Currency_bal,Fiscal Year_bal,Fiscal Period_bal,Publish Date_bal,Restated Date_bal,Shares (Basic)_bal,Shares (Diluted)_bal,"Cash, Cash Equivalents & Short Term Investments",Accounts & Notes Receivable,Inventories,Total Current Assets,"Property, Plant & Equipment, Net",Long Term Investments & Receivables,Other Long Term Assets,Total Noncurrent Assets,Total Assets,Payables & Accruals,Short Term Debt,Total Current Liabilities,Long Term Debt,Total Noncurrent Liabilities,Total Liabilities,Share Capital & Additional Paid-In Capital,Treasury Stock,Retained Earnings,Total Equity,Total Liabilities & Equity
str,i64,date,f64,f64,f64,f64,f64,i64,f64,i64,i64,str,i64,str,date,date,date,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64,str,i64,str,date,date,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""AAPL""",111052,2025-03-10,235.54,236.16,224.22,227.48,226.51,71451281,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31,15081724000,15150865000,53775000000,29639000000,6911000000,133240000000,46069000000,87593000000,77183000000,210845000000,344085000000,61910000000,1995000000,133517000000,94804000000,143810000000,277327000000,84768000000,null,-11221000000,66758000000,344085000000
"""AAPL""",111052,2025-03-11,223.81,225.84,217.45,220.84,219.9,76137410,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31,15081724000,15150865000,53775000000,29639000000,6911000000,133240000000,46069000000,87593000000,77183000000,210845000000,344085000000,61910000000,1995000000,133517000000,94804000000,143810000000,277327000000,84768000000,null,-11221000000,66758000000,344085000000
"""AAPL""",111052,2025-03-12,220.14,221.75,214.91,216.98,216.05,62547467,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31,15081724000,15150865000,53775000000,29639000000,6911000000,133240000000,46069000000,87593000000,77183000000,210845000000,344085000000,61910000000,1995000000,133517000000,94804000000,143810000000,277327000000,84768000000,null,-11221000000,66758000000,344085000000
"""AAPL""",111052,2025-03-13,215.95,216.84,208.42,209.68,208.78,61368330,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31

## Feature Engineering

We engineer 70+ features across four categories:
- **Trend:** Distance from SMA-20, SMA-50, SMA-200
- **Momentum:** RSI proxy (rolling ratio of positive moves)
- **Volatility:** 20-day rolling std, 14-day Average True Range
- **Fundamentals:** Return on Assets, Net Margin (forward-filled quarterly)

In [3]:
# Indicator Battery (Technical + Fundamental)
df = df.with_columns([
    # Normalization
    (pl.col("Close") / pl.col("Close").shift(1)).log().alias("Log_Return_1d"),

    # Trend (Distance from SMAs)
    (pl.col("Close") / pl.col("Close").rolling_mean(20) - 1).alias("Dist_SMA_20"),
    (pl.col("Close") / pl.col("Close").rolling_mean(50) - 1).alias("Dist_SMA_50"),
    (pl.col("Close") / pl.col("Close").rolling_mean(200) - 1).alias("Dist_SMA_200"),

    # Momentum (RSI Proxy)
    ((pl.col("Close").diff() > 0).cast(pl.Int32).rolling_mean(14) /
     (pl.col("Close").diff().abs().rolling_mean(14) + 1e-9)).alias("RSI_Proxy"),

    # Volatility
    pl.col("Close").rolling_std(20).alias("Vol_20"),
    (pl.col("High") - pl.col("Low")).rolling_mean(14).alias("ATR_14"),

    # Fundamental Ratios
    (pl.col("Net Income") / pl.col("Total Assets")).alias("ROA"),
    (pl.col("Net Income") / pl.col("Revenue")).alias("Net_Margin"),
])
print("Features calculated.")

Features calculated.


## Target Engineering and Cleaning

In [4]:
# Target: what happens tomorrow?
df = df.with_columns([
    pl.col("Log_Return_1d").shift(-1).alias("Target_Log_Return"),
    (pl.col("Close").shift(-1) > pl.col("Close")).cast(pl.Int64).alias("Target_Is_Up")
])

# Drop rows missing SMA_200 (start of history) and Target (end of history)
df = df.drop_nulls(subset=["Dist_SMA_200", "Target_Log_Return"])

# Forward-fill fundamentals (quarterly data), fill remaining with 0
df = df.with_columns([
    pl.col("ROA").fill_null(strategy="forward"),
    pl.col("Net_Margin").fill_null(strategy="forward"),
]).fill_null(0.0)

print(f"Final dataset: {df.shape}")
df.select(["Date", "Close", "Log_Return_1d", "Dist_SMA_200", "RSI_Proxy", "ATR_14", "ROA", "Net_Margin", "Target_Is_Up", "Target_Log_Return"]).tail()

Final dataset: (1038, 77)


Date,Close,Log_Return_1d,Dist_SMA_200,RSI_Proxy,ATR_14,ROA,Net_Margin,Target_Is_Up,Target_Log_Return
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-03-07,239.07,0.015768,0.052772,0.179147,5.482857,0.105584,0.292277,0.0,-0.049694
2025-03-10,227.48,-0.049694,0.000931,0.127,6.097143,0.105584,0.292277,0.0,-0.029624
2025-03-11,220.84,-0.029624,-0.028894,0.0877,6.492857,0.105584,0.292277,0.0,-0.017633
2025-03-12,216.98,-0.017633,-0.046414,0.061843,6.803571,0.105584,0.292277,0.0,-0.034223
2025-03-13,209.68,-0.034223,-0.078958,0.054025,7.157143,0.105584,0.292277,1.0,0.018007


## EDA Summary

In [5]:
# Descriptive statistics for key features
key_features = ["Log_Return_1d", "Dist_SMA_20", "Dist_SMA_50", "Dist_SMA_200",
                "RSI_Proxy", "Vol_20", "ATR_14", "ROA", "Net_Margin",
                "Target_Log_Return", "Target_Is_Up"]
df.select(key_features).describe()

statistic,Log_Return_1d,Dist_SMA_20,Dist_SMA_50,Dist_SMA_200,RSI_Proxy,Vol_20,ATR_14,ROA,Net_Margin,Target_Log_Return,Target_Is_Up
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1038.0,1038.0,1038.0,1038.0,1038.0,1038.0,1038.0,1038.0,1038.0,1038.0,1038.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.000369,0.005483,0.014948,0.069861,0.28298,4.827905,3.500137,0.070312,0.25102,0.000385,0.524085
"""std""",0.016764,0.04023,0.062314,0.097225,0.121165,1.874929,1.032394,0.014706,0.028096,0.016773,0.49966
"""min""",-0.060472,-0.115194,-0.15351,-0.182177,0.054025,1.137835,1.712143,0.039128,0.15523,-0.060472,0.0
"""25%""",-0.008335,-0.022303,-0.03199,0.011141,0.1875,3.390827,2.682857,0.05934,0.246533,-0.008335,0.0
"""50%""",0.000787,0.00811,0.015113,0.079107,0.262238,4.699168,3.325,0.070051,0.256497,0.000787,1.0
"""75%""",0.009964,0.033374,0.061776,0.14077,0.357143,5.933046,4.175,0.081216,0.263775,0.010013,1.0
"""max""",0.085236,0.107575,0.175634,0.348401,0.702165,11.926659,7.157143,0.105584,0.292277,0.085236,1.0


In [6]:
# Correlation of engineered features with the regression target
corr_features = ["Dist_SMA_20", "Dist_SMA_50", "Dist_SMA_200", "RSI_Proxy",
                 "Vol_20", "ATR_14", "ROA", "Net_Margin", "Target_Log_Return"]
df.select(corr_features).to_pandas().corr()["Target_Log_Return"].drop("Target_Log_Return").sort_values()

Dist_SMA_200   -0.062493
ATR_14         -0.036730
Dist_SMA_50    -0.029181
ROA            -0.020310
Dist_SMA_20    -0.012649
Net_Margin     -0.010218
Vol_20          0.006366
RSI_Proxy       0.051312
Name: Target_Log_Return, dtype: float64

## Next Steps

This single-ticker pipeline was generalized into `src/etl.py`, which:
- Processes all tickers in a loop to avoid cross-contamination
- Joins company metadata (IndustryId, Number of Employees) after concatenation
- Writes the combined feature set to `data/processed/features.parquet`